# 03 - Data Preparation

## Objetivo

Esta etapa tem como objetivo preparar o conjunto de dados para as etapas posteriores de Engenharia de Atributos e Modelagem de Machine Learning.

Com base nos resultados obtidos durante a Análise Exploratória de Dados (EDA), serão avaliadas e tratadas inconsistências, registros inválidos, valores extremos e variáveis que possam comprometer a construção dos modelos preditivos.

As decisões tomadas nesta etapa serão documentadas e justificadas com base nas evidências encontradas durante a exploração dos dados.

## 1. Carregamento dos dados

In [48]:
from pathlib import Path
import pandas as pd

# Caminho do projeto
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Carregamento dos dados
csv_file = DATA_PATH / "data.csv"
df = pd.read_csv(csv_file)

# Cópia do dataset original para manipulação
df_prep = df.copy()

## 2. Tratamento de área

Durante a EDA, foram identificados imóveis com áreas extremamente reduzidas. Considerando as referências de mercado utilizadas para contextualizar o problema, adotou-se 15 m² como limite inferior plausível para os imóveis presentes no conjunto de dados.

Dessa forma, registros com `area < 15` m² foram considerados inconsistentes e removidos.

A decisão foi tomada com base na combinação entre a análise exploratória dos dados e conhecimento de domínio sobre o mercado imobiliário, evitando a manutenção de registros cuja área provavelmente representa erros de coleta ou cadastro.

In [49]:
df_prep = df_prep[df_prep["area"] >= 15].copy()

## 3. Investigando tipos de Apartamentos e Studio & Kitnets

### 3.1 Apartamentos com 0 quartos

In [50]:
apartamentos_sem_quarto = df_prep[
    (df_prep["type"] == "Apartamento") &
    (df_prep["bedrooms"] == 0)
]

apartamentos_sem_quarto

,address,district,area,bedrooms,garage,type,rent,total
476,Rua Doutor Miguel Vieira Ferreira,Tatuapé,30,0,0,Apartamento,1840,2113
1002,Rua Natividade Saldanha,São Lucas,28,0,0,Apartamento,1100,1391
1416,Rua Natividade Saldanha,São Lucas,27,0,0,Apartamento,1100,1387
1552,Rua Natividade Saldanha,São Lucas,40,0,0,Apartamento,1300,1706
1595,Rua Natividade Saldanha,São Lucas,45,0,0,Apartamento,1400,1855
1925,Rua Santa Lúcia,Cidade Mãe do Céu,22,0,0,Apartamento,1000,1263
9674,Rua Adalberto Kemeny,Parque Industrial Tomas Edson,47,0,1,Apartamento,2330,3190


In [51]:
mask_apartamento_studio = (
    (df_prep["type"] == "Apartamento") &
    (df_prep["bedrooms"] == 0) &
    (df_prep["area"] <= 47)
)

df_prep.loc[mask_apartamento_studio, "type"] = "Studio e kitnet"

### 3.2 Studios & Kitnets com mais de 1 quarto

In [52]:
studios_multiquartos = df_prep[
    (df_prep["type"] == "Studio e kitnet") &
    (df_prep["bedrooms"] > 1)
]

studios_multiquartos

,address,district,area,bedrooms,garage,type,rent,total
272,Rua Leôncio da Costa Vieira,Jardim Piqueroby,41,2,0,Studio e kitnet,2900,3142
491,Rua Antônio Gomes Ferreira,Parque Fongaro,34,2,0,Studio e kitnet,1100,1360
823,Rua Avanhandava,Consolação,35,2,0,Studio e kitnet,1700,2197
1711,Rua Leôncio da Costa Vieira,Jardim Piqueroby,39,2,0,Studio e kitnet,3000,3246
1833,Rua Moisés Marx,Vila Aricanduva,38,2,0,Studio e kitnet,1850,2091
1890,Rua Coronel Gustavo Santiago,Vila Zilda,44,2,0,Studio e kitnet,2550,2799
2089,Rua Catumbi,Catumbi,48,2,1,Studio e kitnet,3000,3539
2094,Rua Leôncio da Costa Vieira,Jardim Piqueroby,36,2,0,Studio e kitnet,2800,3038
2570,Rua Antônio Gomes Ferreira,Parque Fongaro,33,2,0,Studio e kitnet,1200,1627
2727,Rua Santo Antônio,Bela Vista,80,2,1,Studio e kitnet,4500,6284


In [53]:
mask_studio_apartamento = (
    (df_prep["type"] == "Studio e kitnet") &
    (df_prep["bedrooms"] >= 2) &
    (df_prep["area"] > 47)
)

df_prep.loc[mask_studio_apartamento, "type"] = "Apartamento"

In [54]:
studios_multiquartos = df_prep[
    (df_prep["type"] == "Studio e kitnet") &
    (df_prep["bedrooms"] > 1)
]

studios_multiquartos

,address,district,area,bedrooms,garage,type,rent,total
272,Rua Leôncio da Costa Vieira,Jardim Piqueroby,41,2,0,Studio e kitnet,2900,3142
491,Rua Antônio Gomes Ferreira,Parque Fongaro,34,2,0,Studio e kitnet,1100,1360
823,Rua Avanhandava,Consolação,35,2,0,Studio e kitnet,1700,2197
1711,Rua Leôncio da Costa Vieira,Jardim Piqueroby,39,2,0,Studio e kitnet,3000,3246
1833,Rua Moisés Marx,Vila Aricanduva,38,2,0,Studio e kitnet,1850,2091
1890,Rua Coronel Gustavo Santiago,Vila Zilda,44,2,0,Studio e kitnet,2550,2799
2094,Rua Leôncio da Costa Vieira,Jardim Piqueroby,36,2,0,Studio e kitnet,2800,3038
2570,Rua Antônio Gomes Ferreira,Parque Fongaro,33,2,0,Studio e kitnet,1200,1627
2962,Rua Doutor Ângelo Vita,Tatuapé,26,2,0,Studio e kitnet,2200,2744
3242,Rua Glicério,Liberdade,45,2,1,Studio e kitnet,2183,2896


### Regra de reclassificação baseada em área e número de quartos

Durante a investigação da consistência entre `type`, `area` e `bedrooms`, observou-se que alguns imóveis classificados como `Apartamento` apresentavam zero quartos e áreas de até 47 m². Esses registros foram considerados potencialmente compatíveis com a categoria `Studio e kitnet` e reclassificados.

A análise também identificou imóveis classificados como `Studio e kitnet` com dois quartos. Considerando que a existência de dois quartos não implica necessariamente que o imóvel seja um apartamento, adotou-se o limite de 47 m² observado na investigação anterior como critério adicional de classificação.

Assim, Studios e kitnets com dois quartos e área de até 47 m² foram mantidos como `Studio e kitnet`. Já os registros com dois quartos e área superior a 47 m² foram reclassificados como `Apartamento`.

O limite de 47 m² representa uma regra empírica adotada no contexto deste projeto, baseada na análise exploratória dos próprios dados, e não uma regra universal de classificação de imóveis.

In [55]:
pd.crosstab(
    df_prep["type"],
    df_prep["bedrooms"]
)

bedrooms,0,1,2,3,4,5,6
type,,,,,,,
Apartamento,0,2039,3426,1482,227,10,3
Casa,4,757,797,891,291,75,19
Casa em condomínio,0,64,96,49,24,6,1
Studio e kitnet,32,1301,25,0,0,0,0


In [56]:
df_prep.groupby("type")["area"].describe()

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
Apartamento,7187.0,73.436761,50.823144,15.0,45.0,60.0,81.0,568.0
Casa,2834.0,136.456246,101.715610,15.0,60.0,110.0,181.0,580.0
Casa em condomínio,240.0,119.908333,113.181825,16.0,50.0,75.0,141.5,560.0
Studio e kitnet,1358.0,31.782032,9.203484,15.0,25.0,30.0,37.0,92.0


In [57]:
df_prep.groupby("type")["area"].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

type                    
Apartamento         0.01     25.00
                    0.05     30.00
                    0.25     45.00
                    0.50     60.00
                    0.75     81.00
                    0.95    172.00
                    0.99    296.42
Casa                0.01     22.00
                    0.05     30.65
                    0.25     60.00
                    0.50    110.00
                    0.75    181.00
                    0.95    350.00
                    0.99    494.37
Casa em condomínio  0.01     20.78
                    0.05     29.90
                    0.25     50.00
                    0.50     75.00
                    0.75    141.50
                    0.95    380.25
                    0.99    529.76
Studio e kitnet     0.01     15.00
                    0.05     20.00
                    0.25     25.00
                    0.50     30.00
                    0.75     37.00
                    0.95     48.00
                    0.99     6

### Investigação da distribuição de área por tipo de imóvel

A análise dos quantis da variável `area` por categoria revelou diferenças importantes entre os tipos de imóveis. A categoria `Studio e kitnet` apresenta áreas significativamente menores, com mediana de 30 m² e 75% dos registros concentrados em imóveis de até 37 m². Além disso, 95% dos Studios e kitnets possuem até 48 m².

Os apartamentos apresentam uma distribuição mais ampla, com mediana de 60 m², enquanto casas e casas em condomínio possuem áreas medianas ainda maiores, de 110 m² e 75 m², respectivamente.

Apesar da diferença entre as distribuições, observa-se uma sobreposição significativa entre `Studio e kitnet` e `Apartamento`, principalmente na faixa entre aproximadamente 30 m² e 60 m². Dessa forma, a variável `area` isoladamente não é suficiente para determinar a classificação do imóvel.

Com base nessa análise, optou-se por utilizar a área em conjunto com a variável `bedrooms` para identificar inconsistências específicas, evitando uma reclassificação generalizada baseada exclusivamente em um limite de área.

O limite empírico de 47 m² utilizado nas regras de reclassificação encontra-se próximo ao percentil 95 da distribuição de área dos Studios e kitnets, mas não deve ser interpretado como uma fronteira universal entre as categorias.

## 4. Casa com 0 quartos

In [58]:
casas_sem_quarto = df_prep[
    (df_prep["type"] == "Casa") &
    (df_prep["bedrooms"] == 0)
]

casas_sem_quarto

,address,district,area,bedrooms,garage,type,rent,total
1596,Rua Marilândia,Freguesia do Ó,70,0,0,Casa,3100,3152
9501,Rua Dom Armando Lombardi,Vila Progredior,250,0,3,Casa,10000,10290
10062,Rua Graúna,Vila Uberabinha,141,0,0,Casa,7000,7208
10082,Rua Conde de Porto Alegre,Campo Belo,200,0,4,Casa,12500,12790


In [59]:
mask_casa_sem_quarto = (
    (df_prep["type"] == "Casa") &
    (df_prep["bedrooms"] == 0)
)

df_prep = df_prep.loc[~mask_casa_sem_quarto].copy()

### Tratamento de casas com zero quartos

Durante a análise da consistência entre as variáveis `type` e `bedrooms`, foram identificados 4 registros classificados como `Casa` com zero quartos.

Diferentemente dos imóveis classificados como `Apartamento` com zero quartos, para os quais foi levantada a possibilidade de serem Studios ou kitnets, não foi identificada uma classificação alternativa plausível para os registros do tipo `Casa`.

Considerando que a informação de número de quartos apresenta uma inconsistência para esse tipo de imóvel e que não há dados disponíveis para determinar ou recuperar o valor correto, optou-se pela remoção desses 4 registros. A remoção representa uma parcela muito pequena do conjunto de dados e, portanto, espera-se que seu impacto sobre a distribuição geral das observações seja limitado.

## 5. Valores extremos de aluguel

In [61]:
df_prep[df_prep["rent"] >= 15000].sort_values("rent", ascending=False)

,address,district,area,bedrooms,garage,type,rent,total
6095,Avenida Chibarás,Planalto Paulista,24,1,0,Studio e kitnet,25000,26710
400,Rua Helena,Vila Olímpia,126,3,3,Apartamento,15000,17410
1354,Rua Michigan,Cidade Monções,158,3,3,Apartamento,15000,19530
1656,Rua Caminha de Amorim,Alto de Pinheiros,151,2,0,Casa,15000,16140
3565,Avenida Jandira,Indianópolis,212,3,2,Apartamento,15000,17250
...,...,...,...,...,...,...,...,...
11591,Rua Leonardo Cerveira Varandas,Paraíso do Morumbi,300,3,3,Apartamento,15000,18580
11599,Rua Forte William,Jardim Fonte do Morumbi,232,4,5,Apartamento,15000,19520
11603,Rua Domênico Perotti,Jardim Fonte do Morumbi,280,4,5,Apartamento,15000,23180
11637,Alameda Calicut,Chácara Santo Antônio,430,4,4,Casa,15000,17200


In [63]:
rent_15000 = df_prep[df_prep["rent"] == 15000]

rent_15000[
    ["area", "bedrooms", "garage", "rent"]
].describe()

,area,bedrooms,garage,rent
count,113.000000,113.000000,113.000000,113.0
mean,276.424779,3.362832,3.442478,15000.0
std,126.479657,0.991589,1.505573,0.0
min,15.000000,1.000000,0.000000,15000.0
25%,180.000000,3.000000,2.000000,15000.0
50%,260.000000,3.000000,4.000000,15000.0
75%,365.000000,4.000000,5.000000,15000.0
max,560.000000,6.000000,6.000000,15000.0


In [71]:
df_prep[
    (df_prep["rent"] == 15000) &
    (df_prep["area"] <= 15)
]

,address,district,area,bedrooms,garage,type,rent,total
4849,Rua Doutor Alvares Teixeira,Jardim Tango,15,1,1,Casa,15000,15640


In [72]:
df_prep = df_prep.drop(index=4849)

In [62]:
df_prep[df_prep["rent"] == 25000]

,address,district,area,bedrooms,garage,type,rent,total
6095,Avenida Chibarás,Planalto Paulista,24,1,0,Studio e kitnet,25000,26710


In [73]:
df_prep = df_prep.drop(index=6095)

In [74]:
df_prep = df_prep.reset_index(drop=True)

### Tratamento de registro inconsistente com aluguel de R$ 15.000 e R$ 25.000

Durante a investigação dos 113 imóveis com aluguel igual a R$ 15.000, foram analisadas as características de área, número de quartos e vagas de garagem. Foi identificado um registro (nº 4849) classificado como Casa com 15 m² de área, 1 quarto e 1 vaga de garagem. Considerando suas características, o imóvel apresenta uma combinação atípica e potencialmente inconsistente para um aluguel mensal de R$ 15.000.

Embora valores elevados de aluguel possam ser plausíveis no contexto do mercado imobiliário de São Paulo, a combinação entre área extremamente reduzida e aluguel elevado foi considerada pouco representativa para o objetivo de construção de um modelo generalizável. Dessa forma, optou-se pela remoção individual deste registro, utilizando seu identificador, sem remover os demais imóveis com aluguel de R$ 15.000.

Além disso, o imóvel com aluguel de R$ 25.000 também se mostrou inconsistente, dado que é um Studio e kitnet com apenas 24 m².

## 6. Tratamento de Duplicatas

In [77]:
df_prep["address"].duplicated().sum()

np.int64(6281)

In [82]:
df_prep.duplicated(
    subset=[
        "address",
        "area",
        'district',
        "bedrooms",
        "garage",
        "type"
    ]
).sum()

np.int64(3)

In [84]:
df_prep[
    df_prep.duplicated(
        subset=[
            "address",
            "district",
            "area",
            "bedrooms",
            "garage",
            "type"
        ],
        keep=False
    )
].sort_values(
    by=[
        "address",
        "district",
        "area",
        "bedrooms",
        "garage",
        "type"
    ]
)

,address,district,area,bedrooms,garage,type,rent,total
408,Rua Catumbi,Catumbi,48,2,1,Apartamento,1900,2474
2079,Rua Catumbi,Catumbi,48,2,1,Apartamento,3000,3539
1323,Rua Glicério,Liberdade,53,2,1,Apartamento,2450,3375
8318,Rua Glicério,Liberdade,53,2,1,Apartamento,2125,3074
3472,Rua Oliveira Catrambi,Jardim Vila Formosa,50,2,0,Apartamento,6000,6232
6691,Rua Oliveira Catrambi,Jardim Vila Formosa,50,2,0,Apartamento,6000,6577


Não foram identificadas duplicatas exatas no dataset. Uma investigação adicional considerando a combinação de `address`, `district`, `area`, `bedrooms`, `garage` e `type` encontrou apenas 6 registros potencialmente duplicados, organizados em três pares. Entretanto, as diferenças observadas nos valores de aluguel e nos custos totais sugerem que esses registros podem representar imóveis distintos localizados no mesmo endereço, como diferentes unidades de um mesmo edifício. Como não há evidências suficientes para confirmar duplicidade, os registros foram mantidos.

## 7. Integridado dos dados

In [85]:
df_prep.isna().sum()

address     0
district    0
area        0
bedrooms    0
garage      0
type        0
rent        0
total       0
dtype: int64

In [86]:
df_prep[df_prep["area"] < 15]

,address,district,area,bedrooms,garage,type,rent,total


In [87]:
df_prep[df_prep["bedrooms"] < 0]

,address,district,area,bedrooms,garage,type,rent,total


In [88]:
df_prep[df_prep["garage"] < 0]

,address,district,area,bedrooms,garage,type,rent,total


In [89]:
df_prep[df_prep["rent"] <= 0]

,address,district,area,bedrooms,garage,type,rent,total


In [90]:
df_prep["type"].unique()

<StringArray>
['Studio e kitnet', 'Apartamento', 'Casa em condomínio', 'Casa']
Length: 4, dtype: str

In [91]:
df_prep.to_csv(
    "../data/processed/data_cleaned.csv",
    index=False
)

# Resumo do Data Cleaning

Durante a etapa de preparação dos dados, foram investigadas e tratadas inconsistências identificadas durante a análise exploratória dos dados.

As principais decisões tomadas foram:

- Remoção de imóveis com área inferior a 15 m², considerados incompatíveis com o escopo do conjunto de dados.
- Reclassificação de imóveis entre as categorias `Apartamento` e `Studio e kitnet` com base na combinação entre área e número de quartos.
- Investigação e tratamento de imóveis com zero quartos, considerando o tipo e a área do imóvel.
- Remoção de registros considerados inconsistentes após investigação individual.
- Investigação de valores extremos da variável `rent`, mantendo os registros considerados plausíveis e removendo apenas casos específicos considerados inconsistentes.
- Investigação de possíveis duplicidades utilizando diferentes combinações de atributos. Não foram identificadas duplicatas exatas e os poucos registros potencialmente duplicados foram mantidos por não haver evidências suficientes de que representassem o mesmo imóvel.
- Verificação de valores ausentes e inválidos.
- Verificação da consistência entre as variáveis `rent` e `total`.

Após essas etapas, o dataset foi considerado consistente para o início da etapa de Feature Engineering.